# 17.5 Profiling and Performance

**Prerequisites:** 14.1 Complexity Analysis, 14.2 Python's Real Costs, 15.9 Debugging in Practice  
**Target:** Python 3.12+ (notes flag 3.13/3.14 differences)

### What you'll learn
- 🔴 **Measure first** — and why intuition about Python performance is usually wrong
- `timeit` for microbenchmarks, and the four ways it lies
- `cProfile` and `pstats` — where the time actually goes
- 🔴 **`tottime` vs `cumtime`** — the distinction that makes a profile readable
- Profiling a real bottleneck, fixing it, and measuring the difference
- 🔴 The profiler's own overhead, and how it **distorts** call-heavy code
- `tracemalloc` for memory, `-X importtime` for slow startup
- 🔴 The optimisation hierarchy: algorithm beats data structure beats constant factor
- When not to optimise at all

---

## Measure first

> **"Programmers waste enormous amounts of time thinking about, or worrying about, the speed of
> noncritical parts of their programs."** — Donald Knuth

The full quotation ends *"premature optimisation is the root of all evil"*, and it is usually
misquoted as an excuse to never think about performance. The actual argument is narrower and
more useful: **optimise the part that matters, and find out which part that is by measuring.**

Three questions, in order — skip one and you will waste a day:

1. **Is it too slow?** Against a real requirement, not a feeling.
2. **Where does the time go?** A profiler, not a guess.
3. **What is the cheapest change that helps?** Usually an algorithm (**14.1**), not a trick.

🔴 **Intuition about Python performance is reliably wrong.** **14.2** measured it: `x in list`
vs `x in set` at 20,000 items is **265×**; `list.pop(0)` vs `deque.popleft()` draining 30,000 is
**312×**; string `+=` in a loop vs `str.join` on 40,000 pieces is **652×**. Nobody guesses those
numbers.

In [ ]:
import shutil
import subprocess
import sys
import tempfile
import textwrap
from pathlib import Path

WORK = Path(tempfile.mkdtemp(prefix="py175_"))


def write(name, source):
    (WORK / name).write_text(textwrap.dedent(source).lstrip("\n"), encoding="utf-8")
    return name


def run(name, *flags, label=None):
    """Run a script in a fresh interpreter and return its output."""
    done = subprocess.run([sys.executable, *flags, name], cwd=WORK,
                          capture_output=True, text=True, encoding="utf-8",
                          errors="replace", timeout=600)
    shown = label or f"python {' '.join(flags)} {name}".replace("  ", " ")
    body = (done.stdout + done.stderr).strip() or "(no output)"
    return f"$ {shown}\n" + "-" * 68 + "\n" + body


print("scratch:", WORK)

## `timeit` — microbenchmarks

`timeit` runs a snippet many times and reports the best result. It handles the things people
get wrong by hand: it repeats, it takes the **minimum** rather than the mean (noise only ever
makes things slower), and it disables the garbage collector.

```python
timeit.timeit(stmt, setup, number=100_000)      # total seconds for `number` runs
timeit.repeat(stmt, setup, repeat=5, number=1000)  # 5 independent measurements
```

🔴 **Pass code you want measured as `stmt`, and everything else as `setup`.** Building the test
data inside `stmt` measures the data building.

In [ ]:
print(run(write("bench.py", r"""
    import timeit

    SETUP = "data = list(range(20_000)); lookup = set(data); target = 19_999"

    cases = [
        ("target in data   (list)", "target in data"),
        ("target in lookup (set)", "target in lookup"),
    ]

    print(f"{'operation':28}{'seconds / 1000 runs':>22}")
    print("-" * 50)
    results = {}
    for label, stmt in cases:
        best = min(timeit.repeat(stmt, setup=SETUP, repeat=5, number=1000))
        results[label] = best
        print(f"{label:28}{best:22.6f}")

    slow, fast = results.values()
    print(f"\n  the set is {slow / fast:,.0f}x faster at 20,000 items")

    # 🔴 The classic mistake: measuring the setup instead of the operation
    wrong = min(timeit.repeat("data = list(range(20_000)); 19_999 in data",
                              repeat=3, number=100))
    right = min(timeit.repeat("19_999 in data", setup=SETUP, repeat=3, number=100))
    print(f"\n  with setup INSIDE stmt : {wrong:.6f}s  <- mostly list building")
    print(f"  with setup where it belongs: {right:.6f}s")
""")))

The set lookup is thousands of times faster, and the last two lines show the
most common `timeit` mistake — building the list inside the statement measures **the list
building**, not the lookup.

### 🔴 Four ways `timeit` misleads

| Trap | What happens |
|---|---|
| Setup inside `stmt` | you measure data construction, as above |
| A machine under load | absolute numbers are meaningless; **ratios** survive |
| Too small an `n` | you measure interpreter overhead, not your code |
| Unrealistic data | sorted input, tiny strings, no cache misses |

**14.1** hit this hard enough to rebuild an entire cell: wall-clock timing was too noisy to
teach from, `time.process_time()` has ~14.6 ms resolution on Windows, and the fix was to
**count operations first and time second**. Counting is how you reason; timing is how you check.

## `cProfile` — where the time actually goes

`timeit` compares two things you already suspect. A **profiler** tells you what to suspect.

```bash
python -m cProfile -s tottime myscript.py        # whole program
```

```python
import cProfile, pstats
profiler = cProfile.Profile()
profiler.enable()
...
profiler.disable()
pstats.Stats(profiler).sort_stats("tottime").print_stats(10)
```

The program below builds an index and then looks things up in it. One of those two is the
problem — the profile says which.

In [ ]:
SLOW = r"""
    import cProfile
    import io
    import pstats

    TAGS = [f"build-job-{i}" for i in range(3000)]
    WANTED = [f"build-job-{i}" for i in range(0, 3000, 3)]


    def build_index(tags):
        return {tag: [tag] for tag in tags}


    def lookup_scan(index, wanted):
        # 🔴 a linear scan over the keys of a dict
        for key in index:
            if key == wanted:
                return index[key]
        return []


    def report(index):
        return sum(len(lookup_scan(index, w)) for w in WANTED)


    profiler = cProfile.Profile()
    profiler.enable()
    index = build_index(TAGS)
    total = report(index)
    profiler.disable()

    buffer = io.StringIO()
    pstats.Stats(profiler, stream=buffer).strip_dirs().sort_stats("tottime").print_stats(6)
    print(buffer.getvalue())
    print("matched:", total)
"""

print(run(write("slow.py", SLOW)))

One function dominates completely. `lookup_scan` accounts for essentially all
of the runtime, while `build_index` — the part that *looks* expensive, since it builds 3,000
entries — barely registers.

That is the whole value of a profiler: **it corrects the guess you were about to act on.**

## 🔴 `tottime` vs `cumtime`

The two columns answer different questions, and reading the wrong one sends you to the wrong
function.

| Column | Means | Use it to find |
|---|---|---|
| **`tottime`** | time in this function **excluding** everything it calls | 🔴 **the code to actually change** |
| **`cumtime`** | time in this function **including** everything it calls | which subsystem is responsible |

`cumtime` is always largest at the top of the call tree — `main()` has a `cumtime` of the whole
program, which tells you nothing. Sorting by `tottime` finds the leaf where the time is really
being spent.

Same profile, both sorts:

In [ ]:
print(run(write("sorts.py", r"""
    import cProfile
    import io
    import pstats


    def inner(n):
        total = 0
        for i in range(n):
            total += i * i
        return total


    def middle(n):
        return sum(inner(n) for _ in range(20))


    def outer():
        return middle(3000)


    profiler = cProfile.Profile()
    profiler.enable()
    outer()
    profiler.disable()

    for key in ("cumulative", "tottime"):
        buffer = io.StringIO()
        pstats.Stats(profiler, stream=buffer).strip_dirs().sort_stats(key).print_stats(4)
        print("=" * 64)
        print("sorted by", key)
        print(buffer.getvalue())
""")))

Read the two tables side by side.

**Sorted by `cumulative`**, the top entry is `outer()` — technically true and completely
useless: of course the entry point accounts for all the time.

**Sorted by `tottime`**, the top entry is `inner()`, which is where the loop actually runs. That
is the function to change.

🔴 **Sort by `tottime` to find the work; sort by `cumulative` to find the caller responsible for
it.** Most people only ever learn the default, which is neither.

> Other useful `pstats` calls: `print_callers("inner")` shows who calls it, and
> `print_callees("outer")` shows what it calls. `dump_stats(path)` saves a profile so you can
> compare two runs, or open it in `snakeviz` for a flame graph.

## Fixing it, and measuring the fix

In [ ]:
print(run(write("fixed.py", r"""
    import time

    TAGS = [f"build-job-{i}" for i in range(3000)]
    WANTED = [f"build-job-{i}" for i in range(0, 3000, 3)]
    INDEX = {tag: [tag] for tag in TAGS}


    def lookup_scan(index, wanted):
        for key in index:                       # O(n) - the original
            if key == wanted:
                return index[key]
        return []


    def lookup_direct(index, wanted):
        return index.get(wanted, [])            # O(1) - a dict is already an index


    def report(finder):
        return sum(len(finder(INDEX, w)) for w in WANTED)


    timings = {}
    for name, finder in (("scan  O(n)", lookup_scan), ("direct O(1)", lookup_direct)):
        started = time.perf_counter()
        matched = report(finder)
        timings[name] = time.perf_counter() - started
        print(f"  {name:14} {timings[name] * 1000:8.1f} ms   (matched {matched})")

    slow, fast = timings.values()
    print(f"\n  same answer, {slow / fast:,.0f}x faster")
    print("  🔴 The dict was ALREADY an index. The scan threw that away.")
""")))

A large speed-up, and the change was **deleting a loop** — not a clever trick.
The original code did an O(n) scan over a data structure that is O(1) by construction
(**14.2**).

> This is the usual shape of a real fix: the profiler points at a function, you read it, and the
> problem turns out to be an algorithm rather than a slow line.

## 🔴 The profiler distorts what it measures

`cProfile` records **every function call**. That costs time — and the cost is per *call*, not
per second. So it inflates call-heavy code and leaves loop-heavy code almost untouched.

Which means a profile can make function calls look like the bottleneck **when they are not**.

In [ ]:
print(run(write("overhead.py", r"""
    import cProfile
    import time


    def tiny(x):
        return x % 7


    def call_heavy(n):
        return sum(tiny(i) for i in range(n))       # one function call per item


    def loop_heavy(n):
        total = 0
        for i in range(n):
            total += i % 7                          # one function call in total
        return total


    print(f"  {'workload':34}{'plain':>10}{'profiled':>12}{'overhead':>11}")
    print("  " + "-" * 66)
    for label, function in (("call-heavy  (1 call per item)", call_heavy),
                            ("loop-heavy  (1 call total)", loop_heavy)):
        started = time.perf_counter()
        function(300_000)
        plain = time.perf_counter() - started

        profiler = cProfile.Profile()
        started = time.perf_counter()
        profiler.enable()
        function(300_000)
        profiler.disable()
        profiled = time.perf_counter() - started

        print(f"  {label:34}{plain * 1000:8.1f} ms{profiled * 1000:10.1f} ms"
              f"{profiled / plain:10.2f}x")
""")))

**Several times slower for the call-heavy version; essentially free for the
loop-heavy one.** Same profiler, same machine, same number of items.

🔴 **Consequences worth remembering:**

- A `cProfile` result **over-reports function-call cost**. If a profile says "the time is in
  these thousands of tiny calls", verify with `timeit` before restructuring your code.
- Never use `cProfile` numbers as absolute timings — only as **relative weights within one run**.
- For a low-overhead alternative, use a **sampling** profiler (`py-spy`, `scalene`), which
  interrupts periodically rather than instrumenting every call. `py-spy` can even attach to a
  running process, which makes it the right tool in production.

> **Version note.** `sys.monitoring` (3.12, PEP 669) gives tools much cheaper instrumentation
> hooks — the same change that made modern coverage measurement fast (**15.6**).

## Memory, and startup

Two more dimensions of "slow" that a CPU profiler cannot see.

**`tracemalloc`** answers *"what is holding memory?"* — covered as a debugging tool in
**15.9**, since a leak is usually a bug rather than a performance problem. Take two snapshots
and compare.

In [ ]:
print(run(write("memory.py", r"""
    import tracemalloc

    CACHE = {}


    def handle(request_id):
        CACHE[request_id] = [0] * 1000        # never evicted


    def transient(request_id):
        scratch = [0] * 1000                  # released immediately
        return len(scratch)


    tracemalloc.start()
    before = tracemalloc.take_snapshot()

    for i in range(300):
        handle(i)
        transient(i)

    after = tracemalloc.take_snapshot()

    print("growth by line:")
    for stat in after.compare_to(before, "lineno")[:3]:
        location = str(stat).split(chr(92))[-1]
        print("   ", location)

    current, peak = tracemalloc.get_traced_memory()
    print(f"\n   current {current / 1024:,.0f} KiB | peak {peak / 1024:,.0f} KiB")
""")))

`handle` shows as the growth; `transient` allocated just as much in total and
**does not appear**, because its lists were freed. That distinction is the whole point.

**`-X importtime`** answers *"why does my CLI take two seconds to start?"* — a real problem for
command-line tools, and almost always one slow import.

In [ ]:
lines = run(write("startup.py", "import json, csv, sqlite3, decimal\nprint('ready')\n"),
             "-X", "importtime",
             label="python -X importtime startup.py").splitlines()

# Keep the header, then the ten slowest imports by SELF time.
print(lines[0])
print(lines[1])
print(lines[2])
entries = []
for line in lines[3:]:
    parts = line.split("|")
    if len(parts) == 3 and "import time:" in parts[0]:
        try:
            entries.append((int(parts[0].split(":")[1].strip()), line))
        except ValueError:
            continue

print()
print("the ten slowest imports by SELF time:")
for _, line in sorted(entries, reverse=True)[:10]:
    print("  ", line)

The `self` column is time in **that** import alone; `cumulative` includes
everything it pulled in. Sorting by `self` finds the one module to make lazy — the same
`tottime`/`cumtime` distinction as in a profile.

🔴 The usual fix is to **move a heavy import inside the function that needs it**, so a `--help`
run does not pay for it:

```python
def export_csv(rows):
    import pandas as pd          # only paid for when this runs
    ...
```

## 🔴 The optimisation hierarchy

When something is too slow, the options are not equal. Work down this list, not up:

| Level | Typical gain | Example |
|---|---|---|
| **1. Do it less often** | ∞ | caching, `functools.cache`, batching (**4.4**) |
| **2. Better algorithm** | 10–1000× | O(n²) → O(n log n) (**14.1**) |
| **3. Better data structure** | 10–100× | list → set / dict / deque (**14.2**) |
| **4. Better library** | 2–50× | a stdlib C function instead of a Python loop |
| **5. Micro-optimisation** | 1.1–2× | 🔴 local variable lookups, avoiding attribute access |
| **6. Another language / C extension** | 10–100× | last resort, big cost |

🔴 **Levels 1–3 are almost always where the win is**, and they usually make the code *shorter*.
Level 5 makes code worse to read for a gain the profiler can barely see.

The next cell compares a level-2/3 change against a level-5 one on the same problem.

In [ ]:
print(run(write("hierarchy.py", r"""
    import time

    WORDS = [f"tag-{i % 4000}" for i in range(40_000)]
    QUERIES = [f"tag-{i}" for i in range(0, 4000, 2)]


    def with_list(words, queries):
        known = []
        for word in words:
            if word not in known:               # O(n) membership, inside a loop
                known.append(word)
        return sum(1 for q in queries if q in known)


    def with_list_micro(words, queries):
        # Level 5: hoist the lookups into locals. Same algorithm.
        known = []
        append = known.append
        for word in words:
            if word not in known:
                append(word)
        contains = known.__contains__
        return sum(1 for q in queries if contains(q))


    def with_set(words, queries):
        known = set(words)                      # Level 3: the right structure
        return sum(1 for q in queries if q in known)


    results = {}
    for name, fn in (("list        (level 5 target)", with_list),
                     ("list + micro-optimisation", with_list_micro),
                     ("set         (level 3 change)", with_set)):
        started = time.perf_counter()
        answer = fn(WORDS, QUERIES)
        results[name] = time.perf_counter() - started
        print(f"  {name:30}{results[name] * 1000:9.1f} ms   (answer {answer})")

    baseline = results["list        (level 5 target)"]
    print()
    for name, elapsed in results.items():
        print(f"  {name:30}{baseline / elapsed:8.1f}x vs baseline")
""")))

Read the three numbers, and note that the result is **sharper than the
argument usually made**.

The micro-optimisation — hoisting `append` and `__contains__` into locals, which makes the code
harder to read — bought **nothing at all**. On this run it was fractionally *slower*, which is
noise: the honest reading is that it made no measurable difference.

🔴 **And it could not have.** The cost of that function is the `word not in known` scan on a
list, executed 40,000 times. Hoisting an attribute lookup *around* an O(n²) loop cannot help,
because the loop is the entire cost. Micro-optimisation only ever shaves the constant factor —
and there is no constant factor worth shaving here.

The data-structure change (**14.2**: `in list` vs `in set`) bought a factor of several hundred,
and the code is **shorter**.

> That is the argument for the hierarchy in one measurement: level 5 applied to a level-3
> problem buys **zero**. Find the right level first.

## When not to optimise

| Situation | Why leave it |
|---|---|
| It is fast enough | "fast enough" is a requirement, not an opinion |
| It runs once, at startup | 200 ms you pay daily is not worth a week |
| 🔴 The clarity cost is high | you will read this code far more often than it runs |
| You have not measured | you will optimise the wrong thing |
| It is dominated by I/O | 🔴 no CPU optimisation helps a 200 ms network call — that is **12.4**/**12.5** territory |

> **The I/O point deserves emphasis.** If your program spends 95% of its time waiting on a
> network or a disk, `cProfile` will show almost nothing interesting and every CPU improvement
> is invisible. The fix is **concurrency** (**12**), a cache, or fewer requests — not a faster
> loop.

## Interview Questions

1. **How do you decide what to optimise?** *(measure; profile; is it even too slow?)*
2. **`tottime` vs `cumtime` — which do you sort by, and why?** *(`tottime` to find the code to
   change; `cumtime` to find the responsible caller)*
3. **Your profile says the time is in thousands of tiny function calls. What do you check
   before restructuring?** *(🔴 the profiler's own per-call overhead — verify with `timeit`)*
4. **Why does `timeit` report the minimum rather than the mean?** *(noise only ever adds time)*
5. **A colleague optimises a loop by hoisting attribute lookups into locals. What do you ask?**
   *(what did the profiler say, and is the algorithm right first — levels 1–3 before 5)*
6. **Your CLI takes two seconds to start. How do you find out why?** *(`-X importtime`, sort by
   self time, make the heavy import lazy)*
7. **A service's memory grows over days. Which tool?** *(`tracemalloc`, comparing snapshots
   after warm-up — **15.9**)*
8. **Your program spends 95% of its time waiting on HTTP. What does `cProfile` tell you?**
   *(almost nothing useful — the answer is concurrency or caching, **12**)*
9. **When is a micro-optimisation worth it?** *(a measured hot leaf, after the algorithm is
   right, where the clarity cost is small)*
10. **What is the fastest possible optimisation?** *(not doing the work — caching, or deleting
    the feature)*

In [ ]:
# ---- tidy up ----
shutil.rmtree(WORK, ignore_errors=True)
print("scratch removed:", not WORK.exists())

---

## Common Mistakes & Pitfalls

1. 🔴 **Optimising without measuring.** You will pick the wrong function; everyone does.
2. 🔴 **Sorting a profile by `cumulative` and 'fixing' the top entry.** That is `main()`. Sort by `tottime` to find the code to change.
3. **Trusting `cProfile`'s absolute numbers.** Its overhead is per call, so call-heavy code is inflated several-fold while loop-heavy code is barely touched.
4. **Putting setup inside a `timeit` statement.** You measure the setup.
5. **Comparing timings from a loaded machine.** Ratios survive noise; absolute numbers do not (**14.1**).
6. **Reaching for micro-optimisations before checking the algorithm.** Level 5 buys a small factor; level 3 can buy a hundred.
7. **CPU-profiling an I/O-bound program.** The profile is empty and the answer is concurrency (**12**).
8. **Optimising code that runs once at startup**, or that is already fast enough.
9. **Making code unreadable for an unmeasured gain.** You read it far more often than it runs.
10. **Forgetting the fastest optimisation is not doing the work** — cache it, batch it, or delete it.

## Best Practices

- Ask *is it too slow?* against a real requirement before anything else.
- Profile before changing code; profile again after, and keep both numbers.
- Sort by `tottime` to find the work, `cumtime` to find who caused it.
- Use `timeit` to confirm a specific comparison the profiler suggested.
- Count operations as well as timing them when the machine is noisy (**14.1**).
- Work down the hierarchy: do it less, better algorithm, better structure, better library — and only then micro-optimise.
- Use a sampling profiler (`py-spy`, `scalene`) for production and for call-heavy code.
- Keep `-X importtime` in mind for anything with a command-line interface.
- Write the benchmark into your tests if the performance is a requirement (**15.1**).

## Practice Exercises

Try these before moving on.

1. Profile a script of your own with `python -m cProfile -s tottime`. Was the slowest function the one you expected?
2. 🔴 Take the same profile and sort it by `cumulative`. Which function is at the top, and why is that useless?
3. Reproduce the profiler-overhead measurement with a workload of your own. At what call count does the distortion become large enough to mislead you?
4. Use `timeit` to compare `str.join` against `+=` in a loop for 1,000 / 10,000 / 100,000 pieces. Does the ratio grow? Why (**14.2**)?
5. 🔴 Find code in a project of yours that scans a list where a `set` or `dict` would do. Measure before and after.
6. Run `python -X importtime` on a CLI tool you use. What is the single slowest import, and could it be made lazy?
7. Take a function from **14 Data Structure and Algorithm** with a known complexity and confirm the timing matches the theory at n, 2n and 4n.
8. Write a program that is 95% I/O-bound, profile it with `cProfile`, and explain why the output is unhelpful. Then fix it with `concurrent.futures` (**12.4**).
9. **Interview question:** you are told a report takes 40 seconds and must take 5. Walk through your first hour.

---

## Version notes

| Version | Change |
|---|---|
| **3.12** | 🔴 `sys.monitoring` (PEP 669) — much cheaper instrumentation, which is what makes modern coverage and profiling tools fast (**15.6**) |
| **3.12** | Comprehensions are inlined, so they no longer appear as a separate frame in a profile |
| **3.11** | The "faster CPython" work — 10–60% general speed-up, and adaptive specialising interpreter |
| **3.11** | Zero-cost exception handling: `try` blocks cost nothing when nothing is raised (**6.1**) |
| **3.8+** | `functools.cached_property`; `functools.cache` as a simpler `lru_cache(maxsize=None)` |

> **Beyond the standard library.** `py-spy` (sampling, attaches to a running process),
> `scalene` (CPU + memory + GPU, line-level), `line_profiler` (per-line timings) and `snakeviz`
> (flame graphs from a `pstats` dump) are all worth knowing. All of them read or produce the
> same concepts covered here.

## 17 Tooling, Packaging and Environments — the folder

| Notebook | Covers |
|---|---|
| **17.1** | `venv`, `pip`, `sys.path`, dependencies and specifiers |
| **17.2** | `pyproject.toml` — metadata, dependencies and every tool's configuration |
| **17.3** | building wheels and sdists, installing them, publishing |
| **17.4** | `ruff` — linting and formatting |
| **17.5** | this notebook — profiling and performance |

**The one-sentence version:** *an environment you can rebuild, a file that configures
everything, an artefact you can install, a linter that runs on save, and a profiler you consult
before optimising.*

## Related

- **14.1 Complexity Analysis** — counting operations rather than timing them
- **14.2 Python's Real Costs** — the measured ratios this notebook keeps citing
- **15.6 Testing in Practice** — `sys.monitoring`, and coverage's own overhead
- **15.9 Debugging in Practice** — `tracemalloc` for leaks, and reproducibility under noise
- **12 Concurrency** — the answer when the profile is empty because you are I/O-bound
- **18 Working with APIs** — where I/O-bound really starts to matter